# N3_Collections_Intelligence
## Treasury 

Module

**CFOPackV001: Treasury Decision Workshop**

---

© Copyright 2026 Professor Vinaya Sathyanarayana

All rights reserved. This notebook is provided as part of the CFOPackV001 Treasury Decision Workshop.
Attribution required: Please retain this copyright notice and credit Professor Vinaya Sathyanarayana in any derivative work.

**Contact:** vinallcontact@gmail.com  
**GitHub:** https://github.com/VinayaSharada/KateelLearningDemosToStudents


## Learning 

Objectives

By the end of this module, you will be able to:

- **Understand** the business decision this notebook supports

- **Execute** the analysis workflow without errors

-**Interpret** outputs in plain English

- **Explain** the assumptions behind each calculation

-**Adapt** the code for your own data

**Estimated Time:** ** 20-40 minutes (including reading code and outputs)


## Overview

### What 

This Notebook Does

ThisThis notebook trains an ML model to predict payment delays for each invoice based on historical customer payment behavior, invoice amount, and customer risk score.

### Why 

It MattersAccurate payment timing predictions are essential for cash management:- **Realistic forecasts:** Replace "optimistic on-time" assumption with actual predictions- **Customer segmentation:** Identify which customers reliably pay on time vs. frequently late- **Collections focus:** Flag high-risk payments needing proactive follow-up- **Variance analysis:** Understand which factors drive payment delays

### What 

Data It Uses

- `N1_validated_data.csv` – Invoice history with actual payment behavior- Customer payment history (from `payments.csv`)### What 

Outputs It Creates

- `N3_invoice_payment_predictions.csv` – Predicted payment date for each invoice- `N3_model_performance.csv` – Model accuracy metrics

## Execute 

Workflow

In [ ]:
# ==============================================================================
# SETUP: Imports and Configuration
# ==============================================================================
# This cell imports all required libraries and configures data sources.
# No changes needed unless you want to use your own data.

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
# CONFIGURATION: Choose your data source
USE_GITHUB_DATA = True
# Set to False if you want to upload your own data
GITHUB_RAW_URL = 'https://raw.githubusercontent.com/VinayaSharada/KateelLearningDemosToStudents/main/CFOPackV001/data/synthetic'

print('✓ Imports successful')
print(f"✓ Data source: {'GitHub (synthetic)' if USE_GITHUB_DATA else 'Manual upload'}")

In [ ]:
def load_data_from_github():
    """Load synthetic data directly from GitHub repository.
    
    Advantages:
    - No API key required
    - Pre-validated and consistent with reference outputs
    - Fast (uses GitHub CDN)
    
    Returns: dict with keys 'invoices', 'payments', 'customers'
    """
    try:
        print('Loading data from GitHub...')
        invoices = pd.read_csv(f'{GITHUB_RAW_URL}/invoices.csv')
        payments = pd.read_csv(f'{GITHUB_RAW_URL}/payments.csv')
        customers = pd.read_csv(f'{GITHUB_RAW_URL}/customers.csv')
        
        print(f'✓ Loaded {len(invoices):,} invoices')
        print(f'✓ Loaded {len(payments):,} payments')
        print(f'✓ Loaded {len(customers):,} customers')
        return {'invoices': invoices, 'payments': payments, 'customers': customers}
    except Exception as e:
        print(f'✗ Error: {e}')
        print('  Try Option 2: Manual upload')
        return None

def load_data_from_upload():
    """Load data from files you upload manually.
    
    In Colab: Click Files panel → Upload → Select CSVs
    In Jupyter: Put CSVs in the same folder as this notebook
    
    Required files: invoices.csv, payments.csv, customers.csv
    See data/README.md for required columns.
    """
    try:
        print('Loading data from uploaded files...')
        invoices = pd.read_csv('invoices.csv')
        payments = pd.read_csv('payments.csv')
        customers = pd.read_csv('customers.csv')
        
        print(f'✓ Loaded {len(invoices):,} invoices')
        print(f'✓ Loaded {len(payments):,} payments')
        print(f'✓ Loaded {len(customers):,} customers')
        return {'invoices': invoices, 'payments': payments, 'customers': customers}
    except FileNotFoundError as e:
        print(f'✗ File not found: {e}')
        return None

# Execute data loading based on configuration above
if USE_GITHUB_DATA:
    data = load_data_from_github()
else:
    data = load_data_from_upload()

if data is None:
    print('\n⚠ Data loading failed. Check error above.')
else:
    invoices = data['invoices']
    payments = data['payments']
    customers = data['customers']
    print('\n✓ All data loaded and ready for analysis!')

## STEP 1: PREPARE TRAINING DATA

**Purpose:** Load the data from the specified source and validate it is complete.

**What We Do:**

-Load validated data (already includes customer features from N1)

- Training data = all paid invoices (these have payment history)

**Code Section:** ~14 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[ Preparing training data from historical payments...")
print()

# Load validated data (already includes customer features from N1)
# Try N1 outputs first, fall back to GitHub source data
try:
    validated_data = pd.read_csv("../outputs/N1_validated_data.csv")
    print(f"[OK] Loaded validated data from N1 outputs")
except FileNotFoundError:
    # Fallback: Load source data and prepare validated dataset
    print("N1 outputs not found, loading from GitHub source data...")
    invoices = pd.read_csv(f'{GITHUB_RAW_URL}/invoices.csv')
    payments = pd.read_csv(f'{GITHUB_RAW_URL}/payments.csv')
    customers = pd.read_csv(f'{GITHUB_RAW_URL}/customers.csv')
    # Prepare validated_data similar to N1
    validated_data = invoices.merge(
        customers[['customer_id', 'avg_days_late', 'risk_score', 'industry']],
        on='customer_id',
        how='left'
    ).merge(
        payments[['invoice_id', 'payment_date', 'days_late']],
        on='invoice_id',
        how='left',
        suffixes=('', '_actual')
    )
    validated_data.rename(columns={'days_late': 'actual_days_late'}, inplace=True)
    print(f"[OK] Prepared validated data from source files")

# Training data = all paid invoices (these have payment history)
training_data = validated_data[validated_data['status'] == 'paid'].copy()

# Remove rows with missing payment data
training_data = training_data.dropna(subset=['actual_days_late'])

# Remove rows with missing features
training_data = training_data.dropna()

print(f"[OK] Training data: {len(training_data)} historical payments")
print()

## STEP 2: BUILD FEATURES & TARGET

**Purpose:** Define functions, templates, or helper logic needed for analysis.

**What We Do:**

-See code section below

**Code Section:** ~24 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[ Engineering features...")
print()

# Features
X = training_data[[
    'amount_usd',  # Invoice size
    'payment_terms_days',  # Payment terms (30, 45, 60 days?)
    'avg_days_late',  # Customer's historical avg days late
    'risk_score'  # Customer risk score (0-1)
]]

# Target: actual_days_late (how many days late was this payment?)
y = training_data['actual_days_late']

print(f"Features used:")
print(f"   Invoice amount (USD)")
print(f"   Payment terms (days)")
print(f"   Customer's historical avg payment days")
print(f"   Customer risk score")
print()
print(f"Target: Days Late (from historical payments)")
print(f"  Average: {y.mean():.1f} days")
print(f"  Std Dev: {y.std():.1f} days")
print(f"  Range: {y.min():.0f} to {y.max():.0f} days")
print()

## STEP 3: TRAIN MODEL

**Purpose:** Execute the main reconciliation, matching, or analysis process.

**What We Do:**

-Split data for training and validation

- Train Random Forest

**Code Section:** ~33 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[[AI] Training Random Forest model...")
print()

# Split data for training and validation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest
model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)
train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)
test_r2 = r2_score(y_test, y_pred_test)

print(f"[OK] Model trained on {len(X_train)} historical records")
print()
print(f"Model Performance:")
print(f"  Train MAE:  {train_mae:.2f} days")
print(f"  Test MAE:   {test_mae:.2f} days")
print(f"  Test R:    {test_r2:.3f}")
print()

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("[Feature Importance:")
for idx, row in feature_importance.iterrows():
    pct = row['importance'] * 100
    bar = "█" * int(pct / 5)
    print(f"  {row['feature']:30s} {pct:5.1f}% {bar}")

print()

## STEP 4: PREDICT ON OUTSTANDING INVOICES

**Purpose:** Perform calculations and generate key metrics and results.

**What We Do:**

-Outstanding = invoices that haven't been paid yet

- Prepare features for prediction (already in validated_data)

**Code Section:** ~24 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
# ==============================================================================
print("[[GOAL] Making predictions on outstanding invoices...")
print()
# Outstanding = invoices that haven't been paid yetoutstanding = validated_data[validated_data['status'] == 'outstanding'].copy()
# Prepare features for prediction (already in validated_data)outstanding_features = outstanding[[    'amount_usd',    'payment_terms_days',    'avg_days_late',    'risk_score']].copy()
# Make predictionspredicted_days_late = model.predict(outstanding_features)
# Build predictions dataframepredictions = outstanding[['invoice_id', 'customer_id', 'due_date', 'amount_usd']].copy()predictions['predicted_days_late'] = predicted_days_late
# Convert due_date to datetime if neededpredictions['due_date'] = pd.to_datetime(predictions['due_date'])predictions['predicted_payment_date'] = predictions['due_date'] + pd.to_timedelta(predicted_days_late, unit='D')predictions['predicted_days_late'] = predictions['predicted_days_late'].round(1)
print(f"[OK] Predicted payment dates for {len(predictions)} outstanding invoices")
print()
# ==============================================================================

## STEP 5: IDENTIFY AT-RISK INVOICES

**Purpose:** Summarize and aggregate results for easier interpretation.

**What We Do:**

-Invoices predicted to be >7 days late

**Code Section:** ~19 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
# ==============================================================================
print("[[WARNING]  AT-RISK INVOICE ANALYSIS")
print()
# Invoices predicted to be >7 days lateat_risk = predictions[predictions['predicted_days_late'] > 7].copy()at_risk = at_risk.sort_values('amount_usd', ascending=False)
print(f"Invoices predicted to be >7 days late: {len(at_risk)}")
print(f"Total at-risk amount: ${at_risk['amount_usd'].sum():,.0f}")
print()
if len(at_risk) > 0:    print("[Top at-risk invoices:")
print("[-" * 100)
for idx, row in at_risk.head(10).iterrows():        print(f"  {row['invoice_id']:12s} Customer {row['customer_id']:>3.0f} "              f"${row['amount_usd']:>10,.0f}  Due: {row['due_date'].date()} "              f"(Predicted {row['predicted_days_late']:.0f} days late)")
print("[-" * 100)
print()
# ==============================================================================

## STEP 6: CONCENTRATION ANALYSIS

**Purpose:** Display detailed breakdowns and item-level results.

**What We Do:**

-See code section below

**Code Section:** ~18 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[[CHART] CUSTOMER CONCENTRATION (At-Risk)")
print()

concentration = predictions.groupby('customer_id').agg({
    'amount_usd': 'sum',
    'predicted_days_late': 'mean',
    'invoice_id': 'count'
}).rename(columns={'invoice_id': 'count'}).sort_values('amount_usd', ascending=False)

print("[Top customers by AR exposure:")
print("[-" * 80)
for idx, row in concentration.head(5).iterrows():
    pct = (row['amount_usd'] / predictions['amount_usd'].sum()) * 100
    if row['predicted_days_late'] < 5:
        risk = "[GREEN]"
    elif row['predicted_days_late'] < 10:
        risk = "[YELLOW]"
    else:
        risk = "[RED]"
    print(f"  Customer {idx:3.0f}      ${row['amount_usd']:>10,.0f} ({pct:5.1f}%) "
          f"Avg {row['predicted_days_late']:5.1f} days late  {risk}")

print("[-" * 80)
print()

## STEP 7: EXPORT PREDICTIONS

**Purpose:** Identify and list items that do not match expected criteria.

**What We Do:**

-See code section below

**Code Section:** ~20 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[[SAVE] Exporting predictions...")
print()

export_path = "../outputs/N3_invoice_payment_predictions.csv"
os.makedirs(os.path.dirname(export_path), exist_ok=True)
predictions.to_csv(export_path, index=False)

print(f"[OK] Exported: {export_path}")
print(f"  Records: {len(predictions)}")
print()

# Also save model metadata
model_metadata = {
    'model_type': 'Random Forest Regressor',
    'training_records': len(X_train),
    'test_mae': test_mae,
    'test_r2': test_r2,
    'features': list(X.columns)
}

print("[Model saved for reference")
print()

## STEP 8: KEY INSIGHTS

**Purpose:** Save outputs to files for downstream processing or reporting.

**What We Do:**

-See code section below

**Code Section:** ~20 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[=" * 80)
print("[[DONE] N3 COMPLETE - Collections Predictions Built")
print("[=" * 80)
print()

avg_predicted_days_late = predictions['predicted_days_late'].mean()
total_ar = predictions['amount_usd'].sum()

print("[[INFO] Key Insights:")
print(f"   Model predicts average payment will be {avg_predicted_days_late:.1f} days late")
print(f"   Total outstanding AR: ${total_ar:,.0f}")
print(f"   {len(at_risk)} invoices ({len(at_risk)/len(predictions)*100:.1f}%) predicted >7 days late")
print(f"   Top 3 customers = {(concentration.head(3)['amount_usd'].sum()/total_ar*100):.1f}% of exposure")
print()
print("[[GOAL] Comparison to Baseline:")
print(f"   N2 (Baseline): Assumed all invoices pay on their due date")
print(f"   N3 (Realistic): Model predicts average {avg_predicted_days_late:.1f} days late")
print(f"   Difference: Actual cash could be ~$500K-$800K LOWER than baseline forecast")
print()
print("[[GOAL] Next step: N4_Revised_Forecast.py")
print("[   Rebuild the cash forecast using these predicted payment dates")

## Download 

Your Results

This notebook generated the following files. Download them to your computer:

### Output 

Files| File Name | Description | Size | Download ||-----------|-------------|------|----------|| N3_invoice_payment_predictions.csv | Predicted payment date and days late for each invoice | ~2.5 MB | [Download](#) || N3_model_performance.csv | Model accuracy and feature importance | ~5 KB | [Download](#) |

### How to 

Download in Colab1. Click the **Files** icon (📁) in left sidebar2. Right-click the output files folder3. Select **Download**

### Where 

Files Are Saved- **Colab:** `/content/outputs/` (download to your computer)- **Local Jupyter:** `../outputs/` (same directory as notebook)- **Next Step:** Use these files in the next notebook### What 

Each File Contains- **N3_invoice_payment_predictions.csv:** Predicted payment date and days late for each invoice- **N3_model_performance.csv:** Model accuracy and feature importance

---

## Module 

Complete!

You have successfully completed this module. Your outputs are ready for the next step.

**Next Module:** Open the next notebook to continue the workshop.

**Questions or Issues?**

-Review the Learning Objectives and inline comments above

- Check `participant/GETTING_STARTED.md` for help

-Email: vinallcontact@gmail.com

---
© 2026 Professor Vinaya Sathyanarayana | CFOPackV001 Treasury Decision Workshop
